# Phase B.3b - Extraction d'aspects

Ce notebook intègre les approches fondées sur un modèle de langage

- **Extraction d’aspects (Aspect-Based Sentiment Analysis)** :
Une même revue peut être positive sur certains aspects et négative sur d’autres
(par exemple pour un hôtel : lit, bruit, propreté ; pour un restaurant : nourriture,
service, prix).

## 0. Préparation des modèles

Le model choisie : Llama3.2

La collection Meta Llama 3.2 de grands modèles de langage multilingues (LLM) est un ensemble de modèles génératifs pré-entraînés et optimisés pour les instructions, disponibles en tailles 1 et 3 milliards (texte en entrée/texte en sortie). Les modèles Llama 3.2 optimisés pour le texte uniquement sont conçus pour les cas d'utilisation de dialogues multilingues, notamment la recherche et la synthèse automatiques.

In [35]:
import sys
from langchain_ollama import ChatOllama
from langchain_core.callbacks.manager import CallbackManager
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

def get_local_llm(model_name="llama3.2"):
    """
    Configure et retourne une instance du modèle local via Ollama.
    """
    print(f"Connexion au modèle local '{model_name}' via Ollama...")

    try:
        llm = ChatOllama(
            model=model_name,
            temperature=0,
            callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
        )
        
        return llm

    except Exception as e:
        print(f"\n❌ ERREUR CRITIQUE : Impossible de joindre Ollama.")
        print(f"Détails : {e}")
        print("-" * 40)
        return None

### Prendre des "reviews" pour tests

In [36]:
# import json

# reviews_file = "./reviews.json"
# sample_reviews = []
# with open(reviews_file, 'r', encoding='utf-8') as f:
#     for line in f:
#         sample_reviews.append(json.loads(line))


In [37]:
import json
import random

reviews_file = "../data/raw/yelp_academic_reviews4students.jsonl"

reviews = []
with open(reviews_file, 'r', encoding='utf-8') as f:
    for line in f:
        reviews.append(json.loads(line))


# Prendre 10 reviews aléatoires parmi les filtrées
sample_reviews = random.sample(reviews, min(1, len(reviews)))


---

In [38]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List

# Définition de la structure d'un aspect
class AspectSentiment(BaseModel):
    aspect: str = Field(description="Nom de la catégorie générale")
    sentiment: str = Field(description="Sentiment : 'positif' ou 'négatif'")

# Définition de la liste d'aspects
class ABSAResult(BaseModel):
    aspects: List[AspectSentiment]

    
absa_results = []

llm = get_local_llm()

Connexion au modèle local 'llama3.2' via Ollama...


## 1. Analyse

In [39]:
from langchain_core.prompts import PromptTemplate
import pandas as pd
import json

# Initialisation du parser
parser = JsonOutputParser(pydantic_object=ABSAResult)

# Template conçu pour l'extraction structurée (Partie 3 du sujet)
template_absa = """
Tu es un expert en analyse de données Yelp spécialisé en Aspect-Based Sentiment Analysis (ABSA).

TÂCHE :
Analyse l'avis client fourni et extrais les aspects mentionnés UNIQUEMENT s'ils sont associés à un sentiment explicite. 
Tu dois mapper chaque aspect identifié vers une CATEGORIES GENERALES.

CONTRAINTES :
1. Sentiment : Doit être EXACTEMENT "positif" ou "négatif".
2. Exclusion : Si un aspect est mentionné sans sentiment clair ou est ambigu, NE L'INCLUS PAS.
3. Format : Retourne STRICTEMENT un objet JSON (une liste d'objets). Ne rajoute aucune phrase avant ou après le JSON.

Format de réponse :
[
  {{"aspect": "Nom de la catégorie générale", "sentiment": "positif/négatif"}}
]

Si aucun aspect clair n'est trouvé, retourne : []

AVIS À ANALYSER :
{texte_avis}

RÉPONSE JSON :
"""

prompt_absa = PromptTemplate(
    template=template_absa,
    input_variables=["texte_avis"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain_absa = prompt_absa | llm | parser

print("🚀 Lancement de l'extraction d'aspects (ABSA)...")

absa_results = []

for i, review in enumerate(sample_reviews, 1):
    text = review['text']
    
    try:
        response = chain_absa.invoke({"texte_avis": text})
        
        if isinstance(response, list):
            aspects_list = response
        else:
            aspects_list = response.get('aspects', [])
        
        for item in aspects_list:
            aspect = item.get('aspect', '').strip()
            sentiment = item.get('sentiment', '').strip().lower()
            
            if sentiment in ['positif', 'négatif'] and aspect:
                absa_results.append({
                    "id_review": i,
                    "aspect": aspect,
                    "sentiment": sentiment
                })
        print(f"✅ Avis {i} traité.")

    except Exception as e:
        print(f"⚠️ Erreur sur l'avis {i}: {e}")

df_absa = pd.DataFrame(absa_results)

🚀 Lancement de l'extraction d'aspects (ABSA)...
✅ Avis 1 traité.


## 3. Résultat

In [40]:
# Affichage détaillé par avis
for review_id in df_absa['id_review'].unique():
    print(f"\n{'='*100}")
    print(f"📌 AVIS #{review_id}")
    
    # Afficher le texte complet de l'avis
    review_text = sample_reviews[review_id - 1]['text']
    print(f"\nTEXTE COMPLET :")
    print(f"   {review_text}\n")
    
    # Afficher les aspects identifiés
    print(f"ASPECTS IDENTIFIÉS :")
    subset = df_absa[df_absa['id_review'] == review_id]
    
    if len(subset) == 0:
        print(f"   ℹ️  Aucun aspect clair identifié")
    else:
        for _, row in subset.iterrows():
            sentiment = row['sentiment'].lower()
            print(f"    {row['aspect']}: {sentiment}")

print(f"\n{'='*100}")


📌 AVIS #1

TEXTE COMPLET :
   Save your money! Very overpriced for what you get. Ordered the Santa Cruz Chicken Sandwich which is supposed to have bacon. No bacon could be found. When I complained and showed them there was no bacon they said "sure" I'll do a re-make and took the sandwich out of my hands and dumped it in the trash. They were not happy that I requested what was supposed to be there in the first place. We ordered a single burger, fries, Santa Cruz chicken sandwich and a large soda. It came to $23.21. I could have got the same thing at McDonalds for $5.00. The food was not good, and WAY over priced! Will never be back, and I do not recommend. For the same price you can get table service at a Chili's or Applebee's that has a lot more value.

Ended up taking the Santa Cruz chicken sandwich back for the 2nd time due to the poor taste. The manager did give us back our money for the chicken at $10.50.

ASPECTS IDENTIFIÉS :
    Qualité des produits: négatif
    Prix: négatif
  